# **Versionado de la base de datos.**

Este cuaderno se corre **una sola vez**. Guarda en Google Drive una copia fija de la base del proyecto, para que los cuadernos no dependan de que Kaggle siga disponible, y documenta de dónde sale, cuándo se descargó y su huella digital (SHA-256) para verificar que nadie la modifique.

- **Fuente:** *All Lending Club loan data* (George, 2019), Kaggle, archivo `accepted_2007_to_2018Q4.csv.gz`.
- **Descarga original del proyecto:** 26 de octubre de 2026.
- **Qué se guarda:** las 2.260.701 filas del archivo (incluidas las 33 de resumen, para que los cuadernos reproduzcan la limpieza) y las 22 columnas que usan los cuatro cuadernos, en formato Parquet.
- **Dónde:** `MyDrive/LGD_Base_Versionada/`. Los cuadernos buscan la copia ahí y, si no la encuentran, descargan de Kaggle.

In [ ]:
import pandas as pd
import os
import glob
import json
import hashlib
from datetime import datetime
import kagglehub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CARPETA_VERSIONADA = '/content/drive/MyDrive/LGD_Base_Versionada'
ARCHIVO_VERSIONADO = 'lending_club_accepted_2007_2018Q4.parquet'
FECHA_DESCARGA_ORIGINAL = '2026-10-26'
os.makedirs(CARPETA_VERSIONADA, exist_ok=True)

In [ ]:
# Las 22 columnas que usan los cuatro cuadernos del proyecto
COLUMNAS = ['id', 'loan_status', 'funded_amnt', 'total_rec_prncp', 'recoveries', 'collection_recovery_fee',
            'int_rate', 'annual_inc', 'dti', 'grade', 'term', 'purpose', 'issue_d', 'last_pymnt_d',
            'installment', 'total_pymnt',                                  # cuaderno principal y ventana
            'revol_util',                                                  # revisión de la utilización del crédito
            'fico_range_low', 'fico_range_high', 'emp_length', 'home_ownership', 'verification_status']  # exploratorio LightGBM

In [ ]:
# Descarga desde Kaggle y versión del conjunto (número de versión en la ruta)
carpeta_kaggle = kagglehub.dataset_download('wordsforthewise/lending-club')
candidatos = [a for a in glob.glob(os.path.join(carpeta_kaggle, '**', 'accepted_2007_to_2018*.csv*'), recursive=True) if os.path.isfile(a)]
ruta_origen = sorted(candidatos, key=lambda a: not a.endswith('.gz'))[0]
version_kaggle = next((p for p in ruta_origen.split(os.sep) if p.isdigit()), 'no identificada')
print(ruta_origen, '| versión de Kaggle:', version_kaggle)

In [ ]:
def sha256(ruta, bloque=2 ** 20):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for parte in iter(lambda: f.read(bloque), b''):
            h.update(parte)
    return h.hexdigest()

sha_origen = sha256(ruta_origen)
sha_origen

In [ ]:
df = pd.read_csv(ruta_origen, usecols=COLUMNAS, dtype={'id': str}, low_memory=False)
df.shape

In [ ]:
# Control de integridad: la base completa tiene 2.260.701 filas
if df.shape[0] != 2_260_701:
    raise ValueError(f'La base no tiene las filas esperadas: {df.shape[0]:,}')

In [ ]:
ruta_versionada = os.path.join(CARPETA_VERSIONADA, ARCHIVO_VERSIONADO)
df.to_parquet(ruta_versionada, index=False, compression='zstd')
sha_versionado = sha256(ruta_versionada)
print(f'Guardado: {ruta_versionada} | {os.path.getsize(ruta_versionada) / 2 ** 20:.1f} MB')

In [ ]:
# Verificación: la copia se lee igual que el original
copia = pd.read_parquet(ruta_versionada)
assert copia.shape == df.shape
assert copia['funded_amnt'].sum() == df['funded_amnt'].sum()
assert (copia['loan_status'].value_counts(dropna=False) == df['loan_status'].value_counts(dropna=False)).all()
print('Copia verificada: mismas filas, columnas, montos y estados')

In [ ]:
metadatos = {
    'conjunto': 'All Lending Club loan data (George, 2019)',
    'kaggle': 'wordsforthewise/lending-club',
    'version_kaggle': version_kaggle,
    'archivo_origen': os.path.basename(ruta_origen),
    'sha256_archivo_origen': sha_origen,
    'fecha_descarga_original': FECHA_DESCARGA_ORIGINAL,
    'fecha_creacion_copia': datetime.now().strftime('%Y-%m-%d'),
    'archivo_versionado': ARCHIVO_VERSIONADO,
    'sha256_archivo_versionado': sha_versionado,
    'filas': int(df.shape[0]),
    'columnas': COLUMNAS,
    'pandas': pd.__version__,
}
with open(os.path.join(CARPETA_VERSIONADA, 'metadatos_versionado.json'), 'w') as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2)
metadatos

In [ ]:
# Documento para el repositorio: Versionado_Base_Datos.md (se guarda en la misma carpeta de Drive)
documento = f"""# Versionado de la Base de Datos

| Elemento | Valor |
|---|---|
| Conjunto de datos | *All Lending Club loan data* (George, 2019), Kaggle (`wordsforthewise/lending-club`) |
| Versión del conjunto en Kaggle | {version_kaggle} |
| Archivo de origen | `{metadatos['archivo_origen']}` |
| SHA-256 del archivo de origen | `{sha_origen}` |
| Fecha de descarga original del proyecto | 26 de octubre de 2026 |
| Fecha de creación de la copia versionada | {metadatos['fecha_creacion_copia']} |
| Copia versionada | `{ARCHIVO_VERSIONADO}` (Parquet, compresión zstd) |
| SHA-256 de la copia versionada | `{sha_versionado}` |
| Filas | {df.shape[0]:,} (incluidas las 33 filas de resumen del archivo original) |
| Columnas | {len(COLUMNAS)}: {', '.join(COLUMNAS)} |
| Ubicación | Google Drive del equipo: `MyDrive/LGD_Base_Versionada/` |

## Cómo la usan los cuadernos

Los cuatro cuadernos (`Proyecto_LGD_NotebookFinal`, `Proyecto_LGD_VentanaMaduracion`, `Exploratorio_Reformulacion_RecuperacionBayesiana` y `Exploratorio_Recuperacion_LightGBM`) buscan primero la copia versionada en Google Drive y, solo si no la encuentran, descargan la base de Kaggle. En los dos casos verifican que se carguen las 2.260.701 filas.

## Cómo verificar que la copia no cambió

Calcular el SHA-256 del archivo Parquet y compararlo con el de esta tabla. Si no coincide, la copia fue modificada y debe regenerarse con `Proyecto_LGD_VersionadoBase.ipynb`.

## Uso de los datos

La copia se guarda en el Drive privado del equipo, solo para reproducir el análisis académico. No se publica en el repositorio, porque su redistribución depende de la licencia del conjunto en Kaggle.
"""
with open(os.path.join(CARPETA_VERSIONADA, 'Versionado_Base_Datos.md'), 'w') as f:
    f.write(documento)
print(documento)